# Module 3 — CPU solutions

No model download, GPU or paid service is needed. These are formative exercises with explicit fixtures. They do not count as measured capstone evidence. Run all cells in order; assertions check the worked results.

## 1. Supervision mask

Given token IDs, prompt length and padding positions, construct a completion-only label mask. Explain how full-sequence supervision differs.

In [ ]:
ids=[10,11,12,20,21,0,0];prompt_length=3;attention=[1,1,1,1,1,0,0]
completion_labels=[token if i>=prompt_length and attention[i] else -100 for i,token in enumerate(ids)]
full_labels=[token if attention[i] else -100 for i,token in enumerate(ids)]
assert completion_labels==[-100,-100,-100,20,21,-100,-100]
assert full_labels[:3]==[10,11,12]
print(completion_labels,full_labels)


## 2. Decontaminate training

Remove training rows above a .90 train/test cosine threshold while retaining every frozen test identity. A low cosine is not a proof of independence.

In [ ]:
test_ids=["t1","t2"];train=[("a",[.2,.7]),("b",[.91,.1]),("c",[.9,.4])]
retained=[i for i,scores in train if max(scores)<=.9]
dropped=[{"id":i,"max_cosine":max(scores)} for i,scores in train if max(scores)>.9]
assert retained==["a","c"] and test_ids==["t1","t2"]
print(retained,dropped)


## 3. Blind pairwise judging

Map A/B/TIE verdicts back to model identities after changing answer order. Report wins, losses, ties, invalids and the declared denominator.

In [ ]:
cases=[{"A":"tuned","B":"base","verdict":"A"},{"A":"base","B":"tuned","verdict":"B"},{"A":"tuned","B":"base","verdict":"TIE"},{"A":"base","B":"tuned","verdict":"INVALID"}]
wins=sum(r["verdict"] in {"A","B"} and r[r["verdict"]]=="tuned" for r in cases)
losses=sum(r["verdict"] in {"A","B"} and r[r["verdict"]]=="base" for r in cases)
ties=sum(r["verdict"]=="TIE" for r in cases);invalid=sum(r["verdict"]=="INVALID" for r in cases)
assert (wins,losses,ties,invalid)==(2,0,1,1)
print({"wins":wins,"losses":losses,"ties":ties,"invalid":invalid,"win_rate_all_attempts":wins/len(cases)})


## 4. One-variable experiment

Compare two configurations and reject a purported one-variable ablation that changes rank and learning rate together.

In [ ]:
base={"rank":16,"lr":.0002,"epochs":3,"split":"frozen-v1"}
arm={**base,"lr":.0001}
def changed(a,b):return {k for k in a.keys()|b.keys() if a.get(k)!=b.get(k)}
assert changed(base,arm)=={"lr"}
bad={**arm,"rank":8};assert changed(base,bad)=={"lr","rank"}
print("Valid arm",arm,"invalid changes",changed(base,bad))


## Transfer to the model lab

Open `lab.ipynb`. Identify the corresponding real-data function, record the difference between fixture and model evidence, and explain one failure that a numerical score alone would miss.